# WIA3003: Academic Project II
## QCNN Learning Rate Experiment

Compare 0.005, 0.01 and 0.02 using the learned linear readout. Reuse the saved 0.02 runs from notebook 05. Keep the circuit, inputs, validation split, seeds and early-stopping rules fixed. The outer test set is excluded.

Codex assisted with preparing these cells.


## 1.0 Dataset Preparation


In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from pennylane import numpy as pnp
from sklearn.preprocessing import MinMaxScaler

current_folder = Path.cwd()
project_folder = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

split_path = project_folder / "results" / "qcnn_baseline" / "run_details.json"
split_details = json.loads(split_path.read_text(encoding="utf-8"))

data_path = (
    Path.home()
    / "Downloads"
    / "Orbital, Nasal and Craniofacial Class Pattern.xls"
)

workbook_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if workbook_hash != split_details["dataset_sha256"]:
    raise ValueError(
        "The workbook differs from the one used for the saved QCNN split. "
        "Check the file before reusing the row positions."
    )

df = pd.read_excel(data_path, sheet_name="Raw Data")
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()


In [ ]:
features = split_details["features"]
label_mapping = split_details["label_mapping"]

fit_indices = np.array(split_details["fit_row_positions"], dtype=int)
val_indices = np.array(split_details["validation_row_positions"], dtype=int)

fit_records = df.iloc[fit_indices]
val_records = df.iloc[val_indices]

X_fit_raw = fit_records[features].apply(pd.to_numeric, errors="raise")
X_val_raw = val_records[features].apply(pd.to_numeric, errors="raise")

y_fit_raw = fit_records["Gender"].astype("string").str.strip().map(label_mapping)
y_val_raw = val_records["Gender"].astype("string").str.strip().map(label_mapping)

assert X_fit_raw.shape == (664, 8)
assert X_val_raw.shape == (168, 8)
assert not set(fit_indices).intersection(val_indices)
assert np.isfinite(X_fit_raw.to_numpy()).all()
assert np.isfinite(X_val_raw.to_numpy()).all()
assert not y_fit_raw.isna().any()
assert not y_val_raw.isna().any()

fit_groups = set(X_fit_raw.itertuples(index=False, name=None))
val_groups = set(X_val_raw.itertuples(index=False, name=None))
shared_groups = fit_groups & val_groups

assert not shared_groups, "Identical measurements appear in both partitions."

print("Fitting:", X_fit_raw.shape)
print("Validation:", X_val_raw.shape)
print("Shared measurement groups:", len(shared_groups))


In [ ]:
scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)

X_fit = pnp.array(
    scaler.fit_transform(X_fit_raw),
    requires_grad=False,
)

X_val = pnp.array(
    scaler.transform(X_val_raw),
    requires_grad=False,
)

y_fit = pnp.array(y_fit_raw.to_numpy(dtype=int), requires_grad=False)
y_val = pnp.array(y_val_raw.to_numpy(dtype=int), requires_grad=False)

validation_values = X_val_raw.to_numpy()
clipped_count = (
    (validation_values < scaler.data_min_)
    | (validation_values > scaler.data_max_)
).sum()

print("Validation feature values clipped:", int(clipped_count))


## 2.0 Circuit and Training Settings


In [ ]:
import time

import pennylane as qml
from sklearn.metrics import accuracy_score, f1_score, log_loss

SEEDS = [42, 123, 2026]
BATCH_SIZE = 32
LEARNING_RATE = 0.02
MAX_EPOCHS = 30
PATIENCE = 5


def predict_probability(parameters, inputs, readout):
    # Shape: four wire outputs × number of records.
    z = pnp.stack(qcnn_circuit(inputs, parameters[:13]))

    if readout == "Fixed average":
        probability = (1 - pnp.mean(z, axis=0)) / 2
    else:
        # Coefficients start at -1/8 and bias starts at 0,
        # so the initial probability matches the fixed average.
        coefficients = parameters[13:17]
        bias = parameters[17]
        probability = (
            0.5
            + pnp.sum(coefficients[:, None] * z, axis=0)
            + bias
        )

    return pnp.clip(probability, 1e-7, 1 - 1e-7)


def binary_loss(parameters, inputs, labels, readout):
    probability = predict_probability(parameters, inputs, readout)
    return -pnp.mean(
        labels * pnp.log(probability)
        + (1 - labels) * pnp.log(1 - probability)
    )


In [ ]:
n_qubits = 8
surviving_wires = [1, 3, 5, 7]

def convolution_block(parameters, first, second):
    qml.Rot(*parameters[:3], wires=first)
    qml.Rot(*parameters[3:], wires=second)
    qml.CNOT(wires=[first, second])

def convolution_layer(parameters):
    # Apply the same block across neighbouring pairs.
    for first in [0, 2, 4, 6]:
        convolution_block(parameters, first, first + 1)
    for first in [1, 3, 5]:
        convolution_block(parameters, first, first + 1)

def pooling_layer(angle):
    for source, target in [(0, 1), (2, 3), (4, 5), (6, 7)]:
        qml.CRY(angle, wires=[source, target])

device = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(device, interface="autograd", diff_method="backprop")
def qcnn_circuit(inputs, weights):
    for wire in range(n_qubits):
        qml.RY(inputs[..., wire], wires=wire)
    convolution_layer(weights[:6])
    convolution_layer(weights[6:12])
    pooling_layer(weights[12])
    return [qml.expval(qml.PauliZ(wire)) for wire in surviving_wires]




In [ ]:
def train_readout(seed, readout, learning_rate):
    # Same circuit initialisation for both readouts at each seed.
    rng = np.random.default_rng(seed)
    circuit_weights = rng.uniform(-0.1, 0.1, size=13)

    if readout == "Learned readout":
        initial_weights = np.concatenate(
            [circuit_weights, np.full(4, -1 / 8), [0.0]]
        )
    else:
        initial_weights = circuit_weights

    parameters = pnp.array(initial_weights, requires_grad=True)
    optimiser = qml.AdamOptimizer(stepsize=learning_rate)
    shuffle_rng = np.random.default_rng(seed)

    best_loss = np.inf
    best_epoch = None
    best_parameters = None
    waiting = 0
    history = []
    start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        order = shuffle_rng.permutation(len(X_fit))

        for offset in range(0, len(order), BATCH_SIZE):
            batch = order[offset : offset + BATCH_SIZE]

            def batch_loss(current):
                return binary_loss(
                    current, X_fit[batch], y_fit[batch], readout
                )

            parameters = optimiser.step(batch_loss, parameters)

        fitting_loss = float(binary_loss(parameters, X_fit, y_fit, readout))
        validation_loss = float(binary_loss(parameters, X_val, y_val, readout))

        if not np.isfinite([fitting_loss, validation_loss]).all():
            raise ValueError(f"Non-finite loss: {readout}, seed {seed}, epoch {epoch}")

        history.append({
            "Readout": readout,
            "Seed": seed,
            "Epoch": epoch,
            "Fitting loss": fitting_loss,
            "Validation loss": validation_loss,
        })

        if validation_loss < best_loss:
            best_loss = validation_loss
            best_epoch = epoch
            best_parameters = np.array(parameters, dtype=float, copy=True)
            waiting = 0
        else:
            waiting += 1
            if waiting >= PATIENCE:
                break

    runtime = time.perf_counter() - start

    # Score the best checkpoint, not the last training epoch.
    checkpoint = pnp.array(best_parameters, requires_grad=False)
    probabilities = np.asarray(
        predict_probability(checkpoint, X_val, readout), dtype=float
    )
    predictions = (probabilities >= 0.5).astype(int)
    labels = np.asarray(y_val, dtype=int)

    result = {
        "Readout": readout,
        "Seed": seed,
        "Best epoch": best_epoch,
        "Epochs run": len(history),
        "Validation macro F1": f1_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "Validation accuracy": accuracy_score(labels, predictions),
        "Validation loss": log_loss(labels, probabilities, labels=[0, 1]),
        "Runtime (s)": runtime,
    }

    return result, history, best_parameters


## 3.0 Run Learning Rate Comparison

Two new rates and three seeds give six new runs. Each run saves its best checkpoint and loss history as it finishes. Rerunning this cell skips completed runs in the same output folder. Use a new folder if the split, settings or training code changes.


In [ ]:
import joblib

output_folder = project_folder / "results" / "qcnn_learning_rate"
output_folder.mkdir(parents=True, exist_ok=True)
reference_path = project_folder / "results" / "qcnn_readout_experiment" / "validation_runs.csv"
reference = pd.read_csv(reference_path)
reference = reference.loc[reference["Readout"] == "Learned readout"].copy()
assert sorted(reference["Seed"].tolist()) == sorted(SEEDS), "Expected one saved 0.02 run per seed."
reference["Learning rate"] = 0.02
reference["Source"] = "Notebook 05"

settings = {
    "dataset_sha256": workbook_hash,
    "fit_row_positions": fit_indices.tolist(),
    "validation_row_positions": val_indices.tolist(),
    "features": features,
    "seeds": SEEDS,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rates": [0.005, 0.01, 0.02],
    "reference_sha256": hashlib.sha256(reference_path.read_bytes()).hexdigest(),
}
settings_path = output_folder / "settings.json"
if settings_path.exists():
    assert json.loads(settings_path.read_text(encoding="utf-8")) == settings, "Settings changed; use a new output folder."
else:
    settings_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
joblib.dump(scaler, output_folder / "fitting_scaler.joblib")

new_runs = []
for rate in [0.005, 0.01]:
    for seed in SEEDS:
        stem = output_folder / f"lr_{rate}_seed_{seed}"
        record_path = Path(str(stem) + ".json")
        weights_path = Path(str(stem) + "_weights.npy")
        history_path = Path(str(stem) + "_history.csv")
        if record_path.exists() and weights_path.exists() and history_path.exists():
            result = json.loads(record_path.read_text(encoding="utf-8"))
            assert result["Learning rate"] == rate and result["Seed"] == seed
            print(f"Using saved run: rate {rate}, seed {seed}", flush=True)
        else:
            print(f"Training: rate {rate}, seed {seed}", flush=True)
            result, history, weights = train_readout(seed, "Learned readout", rate)
            result.update({"Learning rate": rate, "Source": "Notebook 06"})
            np.save(weights_path, weights)
            pd.DataFrame(history).assign(**{"Learning rate": rate}).to_csv(history_path, index=False)
            record_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
        new_runs.append(result)
        print(f"  Macro F1: {result['Validation macro F1']:.4f} | Accuracy: {result['Validation accuracy']:.2%} | Best epoch: {result['Best epoch']}", flush=True)
        runs = pd.concat([reference, pd.DataFrame(new_runs)], ignore_index=True)
        runs.to_csv(output_folder / "validation_runs.csv", index=False)
display(runs.sort_values(["Learning rate", "Seed"]).round(4))



## 4.0 Compare Validation Results

Compare mean validation macro F1 first. The standard deviation measures variation across seeds on this split. Also inspect validation loss and accuracy. Keep the 210 outer test records out of the decision.


In [ ]:
assert len(runs) == 9 and not runs.duplicated(["Learning rate", "Seed"]).any()
metrics = ["Validation macro F1", "Validation accuracy", "Validation loss", "Best epoch", "Runtime (s)"]
summary = runs.groupby("Learning rate")[metrics].agg(["mean", "std"])
paired = runs.pivot(index="Seed", columns="Learning rate", values="Validation macro F1")
for rate in [0.005, 0.01]:
    paired[f"{rate} minus 0.02"] = paired[rate] - paired[0.02]
display(paired.round(4))
display(summary.round(4))
paired.to_csv(output_folder / "paired_macro_f1.csv")
summary.to_csv(output_folder / "validation_summary.csv")

